# Notebook 09: Linearity Validation & Biological Force Scaling
**Project**: Stegoceras Biomechanics & Uncertainty Quantification  
**Specimen**: *Stegoceras validum* (UALVP 2, referred specimen)  
**Deliverable**: Phase 4 Linearity & Scaling Validation  

### Objective
Validate exact Hookean linear scaling across multiple load magnitudes ($500\text{ N}$, $1000\text{ N}$, $2000\text{ N}$), proving zero numerical artifact in linear elasticity ($|\epsilon_{\text{lin}}| = 0.000\%$) and quadratic strain energy scaling ($U \propto F^2$), and demonstrating direct analytical mapping to the literature-derived biological load ($F = 1360\text{ N} = 1.36 \times 1.0\text{ kN}$). 

In [1]:
import numpy as np
from stegoceras_biomechanics.fea.meshing import extract_boundary_surface
from stegoceras_biomechanics.fea.loads import generate_dome_load_patch
from stegoceras_biomechanics.fea.boundary_conditions import generate_boundary_constraints
from stegoceras_biomechanics.fea.solver import solve_linear_elasticity
from stegoceras_biomechanics.fea.validation import verify_load_linearity

med_data = np.load('../data/meshes/cleaned/stegoceras_tetmesh_medium.npz')
nodes = med_data['nodes']
elements = med_data['elements']
surf = extract_boundary_surface(nodes, elements)

condyle_nodes, nuchal_nodes, _ = generate_boundary_constraints(surf)

solutions = {}
for F in [500.0, 1000.0, 2000.0]:
    loaded_nodes, nodal_forces, _, _ = generate_dome_load_patch(surf, target_area_mm2=3000.0, force_magnitude_N=F)
    sol = solve_linear_elasticity(
        nodes=nodes,
        elements=elements,
        youngs_modulus_MPa=17000.0,
        poisson_ratio=0.30,
        loaded_node_indices=loaded_nodes,
        nodal_forces_N=nodal_forces,
        condyle_node_indices=condyle_nodes,
        nuchal_node_indices=nuchal_nodes,
        solver_method='direct',
    )
    solutions[F] = sol

lin_res = verify_load_linearity(solutions[500.0], solutions[1000.0], solutions[2000.0])
print('=== Linearity Verification Results ===')
print(f'Max Displacements: {lin_res.max_displacements_mm} mm')
print(f'95th Percentile Stresses: {lin_res.p95_stresses_MPa} MPa')
print(f'Total Strain Energies: {lin_res.total_strain_energies_mJ} mJ')
print(f'Displacement Linearity Error: {lin_res.displacement_linearity_error_pct:.8f}%')
print(f'Stress Linearity Error: {lin_res.stress_linearity_error_pct:.8f}%')
print(f'Energy Quadratic Error: {lin_res.energy_quadratic_error_pct:.8f}%')
assert lin_res.is_linear, 'Linearity validation failed!'

# Biological Scaling (1360 N)
scale_bio = 1360.0 / 1000.0
bio_disp_um = lin_res.max_displacements_mm[1] * scale_bio * 1000.0
bio_stress_p95 = lin_res.p95_stresses_MPa[1] * scale_bio
bio_energy_mj = lin_res.total_strain_energies_mJ[1] * (scale_bio**2)
print('\n=== Derived Biological Impact Load (F = 1360 N) ===')
print(f'Max Displacement (Biological): {bio_disp_um:.2f} μm')
print(f'95th Percentile Stress (Biological): {bio_stress_p95:.2f} MPa')
print(f'Total Strain Energy (Biological): {bio_energy_mj:.4f} mJ')
print('✓ Perfect linear scaling and biological load transformation verified!')

Transforming over 1000 vertices to C_CONTIGUOUS.


Transforming over 1000 elements to C_CONTIGUOUS.


Transforming over 1000 vertices to C_CONTIGUOUS.


Transforming over 1000 elements to C_CONTIGUOUS.


Transforming over 1000 vertices to C_CONTIGUOUS.


Transforming over 1000 elements to C_CONTIGUOUS.


=== Linearity Verification Results ===
Max Displacements: [0.017767544775681956, 0.03553508955136391, 0.07107017910272782] mm
95th Percentile Stresses: [0.7961205388733776, 1.5922410777467553, 3.1844821554935105] MPa
Total Strain Energies: [1.6613574958603885, 6.645429983441554, 26.581719933766216] mJ
Displacement Linearity Error: 0.00000000%
Stress Linearity Error: 0.00000000%
Energy Quadratic Error: 0.00000000%

=== Derived Biological Impact Load (F = 1360 N) ===
Max Displacement (Biological): 48.33 μm
95th Percentile Stress (Biological): 2.17 MPa
Total Strain Energy (Biological): 12.2914 mJ
✓ Perfect linear scaling and biological load transformation verified!
